# Chapter 1

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct',
                                             device_map ='auto',
                                             torch_dtype= 'auto')

tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [2]:
message =[
    {"role" : "user",
     "content" : "hello"}
]

In [3]:
prompt = tokenizer.apply_chat_template(message,
                                       tokenize = False,
                                       add_generation_prompt = True)
print(prompt)

<|user|>
hello<|end|>
<|assistant|>



In [4]:
input = tokenizer(prompt, return_tensors="pt")
print(input)
inputs = {k: v.to(model.device) for k, v in input.items()}

{'input_ids': tensor([[32010, 22172, 32007, 32001]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [5]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

In [6]:
outputs

tensor([[32010, 22172, 32007, 32001, 15043, 29991,  1128,   508,   306,  6985,
           366,  9826, 29973, 32007]], device='cuda:0')

In [7]:
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(response)

Hello! How can I assist you today?


# Chapter 2

In [29]:
message = [
    {"role" : "user",
     "content" : "Write a leave request mail to Hr."}
]

prompt = tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt=True)
print(f"Prompt: {prompt}\n\n")

inputs = tokenizer(prompt, return_tensors="pt")
print(f"input ids: {inputs}\n\n")

#inputs = {k: v.to(model.device) for k, v in inputs.items()}

for k,v in inputs.items():
  inputs[k] = v.to(model.device)

generation_output = model.generate(input_ids = inputs['input_ids'], max_new_tokens = 20)
print(f"output: {generation_output}\n\n")

print(f"Decoded: {tokenizer.decode(generation_output[0])}\n\n")

for token in inputs["input_ids"][0]:
    print(token.item(), "->", tokenizer.decode(token))



Prompt: <|user|>
Write a leave request mail to Hr.<|end|>
<|assistant|>



input ids: {'input_ids': tensor([[32010, 14350,   263,  5967,  2009, 10524,   304, 26399, 29889, 32007,
         32001]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


output: tensor([[32010, 14350,   263,  5967,  2009, 10524,   304, 26399, 29889, 32007,
         32001,  3323,   622, 29901,   951,  1351, 10729,   363,   518, 10858,
          4408, 29962,    13,    13, 29928,   799,   518, 29950,  7889, 27562,
         15629]], device='cuda:0')


Decoded: <|user|> Write a leave request mail to Hr.<|end|><|assistant|> Subject: Leave Request for [Your Name]

Dear [Human Resources Manager


32010 -> <|user|>
14350 -> Write
263 -> a
5967 -> leave
2009 -> request
10524 -> mail
304 -> to
26399 -> Hr
29889 -> .
32007 -> <|end|>
32001 -> <|assistant|>


In [31]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

show_tokens(prompt, "microsoft/Phi-3-mini-4k-instruct")

<|user|> Write a leave request mail to Hr . <|end|> <|assistant|> 

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation",
                     model = model,
                     tokenizer = tokenizer,
                     return_full_text = False,
                     max_new_tokens = 500,
                     do_sample = False)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
